# Predicting the Cause: What Drives Major Power Outages in the U.S.?

**Name(s)**: Shivam Sharma

**Website Link**: (your website link)

In [20]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * 

!pip install openpyxl
import os
print(os.getcwd())

/Users/shivamsharma0608/Desktop/power_outage_causation


## Step 1: Introduction

In [21]:
outages_raw = pd.read_excel('outage.xlsx', skiprows=5, header=0)
outages_raw = outages_raw.drop(index=0).reset_index(drop=True)
outages_raw = outages_raw.drop(columns=['variables'], errors='ignore')
print(f"Shape: {outages_raw.shape}")
print(outages_raw.head(3))
relevant_cols = [
    'YEAR',               # Year outage occurred
    'MONTH',              # Month (1-12)
    'U.S._STATE',         # State where outage occurred
    'NERC.REGION',        # NERC reliability region
    'CLIMATE.REGION',     # Climate region (e.g. Northeast, Southeast)
    'ANOMALY.LEVEL',      # Climate anomaly level (El Niño/La Niña indicator)
    'CLIMATE.CATEGORY',   # Climate episode (Warm/Cold/Normal)
    'CAUSE.CATEGORY',     # *** TARGET: cause of the outage ***
    'CAUSE.CATEGORY.DETAIL', # More detail on cause
    'OUTAGE.DURATION',    # Duration in minutes
    'DEMAND.LOSS.MW',     # Peak demand loss in megawatts
    'CUSTOMERS.AFFECTED', # Number of customers affected
    'TOTAL.PRICE',        # Average electricity price (cents/kWh)
    'TOTAL.SALES',        # Total electricity consumption (MWh)
    'TOTAL.CUSTOMERS',    # Total customers served
    'POPPCT_URBAN',       # % of state population that is urban
    'POPDEN_URBAN',       # Urban population density
    'AREAPCT_URBAN',      # % of state area that is urban
    'OUTAGE.START.DATE',
    'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE',
    'OUTAGE.RESTORATION.TIME',
]
outages_raw

Shape: (1534, 56)
   OBS    YEAR  MONTH U.S._STATE  ... AREAPCT_UC PCT_LAND PCT_WATER_TOT  \
0  1.0  2011.0    7.0  Minnesota  ...        0.6    91.59          8.41   
1  2.0  2014.0    5.0  Minnesota  ...        0.6    91.59          8.41   
2  3.0  2010.0   10.0  Minnesota  ...        0.6    91.59          8.41   

  PCT_WATER_INLAND  
0             5.48  
1             5.48  
2             5.48  

[3 rows x 56 columns]


,OBS,YEAR,MONTH,U.S._STATE,...,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,1.0,2011.0,7.0,Minnesota,...,0.6,91.59,8.41,5.48
1,2.0,2014.0,5.0,Minnesota,...,0.6,91.59,8.41,5.48
2,3.0,2010.0,10.0,Minnesota,...,0.6,91.59,8.41,5.48
...,...,...,...,...,...,...,...,...,...
1531,1532.0,2009.0,8.0,South Dakota,...,0.15,98.31,1.69,1.69
1532,1533.0,2009.0,8.0,South Dakota,...,0.15,98.31,1.69,1.69
1533,1534.0,2000.0,NaN,Alaska,...,0.02,85.76,14.24,2.9


## Step 2: Data Cleaning and Exploratory Data Analysis

In [22]:
outages = outages_raw.copy()

# 1. Combine date + time into single pd.Timestamp columns (required by spec)
outages['OUTAGE.START'] = pd.to_datetime(
    outages['OUTAGE.START.DATE'].astype(str) + ' ' + outages['OUTAGE.START.TIME'].astype(str),
    errors='coerce'
)
outages['OUTAGE.RESTORATION'] = pd.to_datetime(
    outages['OUTAGE.RESTORATION.DATE'].astype(str) + ' ' + outages['OUTAGE.RESTORATION.TIME'].astype(str),
    errors='coerce'
)

# 2. Drop the original split date/time columns
outages = outages.drop(columns=[
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME'
])

# 3. Replace 0s in columns where 0 is meaningless with NaN
#    (e.g. 0 customers affected or 0 demand loss likely means data is missing)
for col in ['DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED', 'OUTAGE.DURATION']:
    outages[col] = outages[col].replace(0, np.nan)

# 4. Extract useful time features from the timestamp
outages['OUTAGE.MONTH'] = outages['OUTAGE.START'].dt.month
outages['OUTAGE.HOUR'] = outages['OUTAGE.START'].dt.hour

# 5. Create a SEASON column from month
def month_to_season(month):
    if pd.isna(month):
        return np.nan
    month = int(month)
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

outages['SEASON'] = outages['OUTAGE.MONTH'].apply(month_to_season)

# 6. Keep only rows where CAUSE.CATEGORY is not null (our target)
outages = outages[outages['CAUSE.CATEGORY'].notna()].reset_index(drop=True)

print(f"Cleaned shape: {outages.shape}")
print(f"\nCause category distribution:\n{outages['CAUSE.CATEGORY'].value_counts()}")
print(f"\nMissing values in key columns:")
print(outages[['CAUSE.CATEGORY','CLIMATE.REGION','ANOMALY.LEVEL',
               'CUSTOMERS.AFFECTED','DEMAND.LOSS.MW','OUTAGE.DURATION']].isna().sum())

# Show head for website
print(outages[['YEAR','U.S._STATE','CLIMATE.REGION','CAUSE.CATEGORY',
               'OUTAGE.DURATION','CUSTOMERS.AFFECTED','OUTAGE.START']].head())


# ── Univariate Analysis ───────────────────────────────────────────────────────

# Plot 1: Distribution of CAUSE.CATEGORY
cause_counts = outages['CAUSE.CATEGORY'].value_counts().reset_index()
cause_counts.columns = ['Cause Category', 'Count']

fig1 = px.bar(
    cause_counts,
    x='Cause Category', y='Count',
    title='Distribution of Power Outage Cause Categories',
    labels={'Cause Category': 'Cause Category', 'Count': 'Number of Outages'},
    color='Count',
    color_continuous_scale='Blues',
    template='plotly_white'
)
fig1.update_layout(showlegend=False, xaxis_tickangle=-30)
fig1.show()
fig1.write_html('assets/cause_distribution.html', include_plotlyjs='cdn')

# Plot 2: Distribution of OUTAGE.DURATION (log scale due to skew)
fig2 = px.histogram(
    outages[outages['OUTAGE.DURATION'].notna()],
    x='OUTAGE.DURATION',
    nbins=60,
    title='Distribution of Outage Duration (minutes)',
    labels={'OUTAGE.DURATION': 'Duration (minutes)'},
    template='plotly_white',
    color_discrete_sequence=['steelblue']
)
fig2.update_layout(yaxis_title='Count')
fig2.show()
fig2.write_html('assets/duration_distribution.html', include_plotlyjs='cdn')


# ── Bivariate Analysis ────────────────────────────────────────────────────────

# Plot 3: Average outage duration by cause category
duration_by_cause = (
    outages.groupby('CAUSE.CATEGORY')['OUTAGE.DURATION']
    .median()
    .reset_index()
    .sort_values('OUTAGE.DURATION', ascending=False)
)
duration_by_cause.columns = ['Cause Category', 'Median Duration (min)']

fig3 = px.bar(
    duration_by_cause,
    x='Cause Category', y='Median Duration (min)',
    title='Median Outage Duration by Cause Category',
    template='plotly_white',
    color='Median Duration (min)',
    color_continuous_scale='Reds'
)
fig3.update_layout(xaxis_tickangle=-30, showlegend=False)
fig3.show()
fig3.write_html('assets/duration_by_cause.html', include_plotlyjs='cdn')

# Plot 4: Cause category breakdown by climate region (stacked bar)
climate_cause = (
    outages.groupby(['CLIMATE.REGION', 'CAUSE.CATEGORY'])
    .size()
    .reset_index(name='Count')
)
fig4 = px.bar(
    climate_cause,
    x='CLIMATE.REGION', y='Count',
    color='CAUSE.CATEGORY',
    title='Cause Category Breakdown by Climate Region',
    labels={'CLIMATE.REGION': 'Climate Region', 'Count': 'Number of Outages'},
    template='plotly_white',
    barmode='stack'
)
fig4.update_layout(xaxis_tickangle=-30, legend_title='Cause Category')
fig4.show()
fig4.write_html('assets/cause_by_climate.html', include_plotlyjs='cdn')


# ── Interesting Aggregates ────────────────────────────────────────────────────

# Pivot table: median customers affected and duration by cause + season
pivot = outages.pivot_table(
    index='CAUSE.CATEGORY',
    columns='SEASON',
    values='CUSTOMERS.AFFECTED',
    aggfunc='median'
)
print("\nMedian Customers Affected by Cause Category and Season:")
print(pivot.round(0))
print(pivot.round(0).to_markdown())  

Cleaned shape: (1534, 57)

Cause category distribution:
CAUSE.CATEGORY
severe weather                   763
intentional attack               418
system operability disruption    127
public appeal                     69
equipment failure                 60
fuel supply emergency             51
islanding                         46
Name: count, dtype: int64

Missing values in key columns:
CAUSE.CATEGORY          0
CLIMATE.REGION          6
ANOMALY.LEVEL           9
CUSTOMERS.AFFECTED    655
DEMAND.LOSS.MW        901
OUTAGE.DURATION       136
dtype: int64
     YEAR U.S._STATE      CLIMATE.REGION      CAUSE.CATEGORY  OUTAGE.DURATION  \
0  2011.0  Minnesota  East North Central      severe weather           3060.0   
1  2014.0  Minnesota  East North Central  intentional attack              1.0   
2  2010.0  Minnesota  East North Central      severe weather           3000.0   
3  2012.0  Minnesota  East North Central      severe weather           2550.0   
4  2015.0  Minnesota  East North Centr


Median Customers Affected by Cause Category and Season:
SEASON                             Fall    Spring    Summer    Winter
CAUSE.CATEGORY                                                       
equipment failure              900000.0   80915.0   45452.0   52000.0
fuel supply emergency               NaN       NaN       NaN       1.0
intentional attack               9200.0    5852.0    1100.0    2500.0
islanding                        7077.0    9700.0     606.0    6635.0
public appeal                       NaN       NaN    8000.0   18600.0
severe weather                 118000.0  102568.0  109000.0  118000.0
system operability disruption  104000.0   82500.0   33500.0   51982.0
| CAUSE.CATEGORY                |   Fall |   Spring |   Summer |   Winter |
|:------------------------------|-------:|---------:|---------:|---------:|
| equipment failure             | 900000 |    80915 |    45452 |    52000 |
| fuel supply emergency         |    nan |      nan |      nan |        1 |
| intenti

## Step 3: Assessment of Missingness

In [23]:
outages['CUSTOMERS.AFFECTED.MISSING'] = outages['CUSTOMERS.AFFECTED'].isna()

print(f"\nCustomers Affected missing rate: {outages['CUSTOMERS.AFFECTED.MISSING'].mean():.2%}")

# --- Test 1: Does missingness of CUSTOMERS.AFFECTED depend on CAUSE.CATEGORY? ---
# Test statistic: TVD (since CAUSE.CATEGORY is categorical)

def tvd(group_a, group_b):
    """Compute Total Variation Distance between two categorical distributions."""
    cats = set(group_a) | set(group_b)
    dist_a = pd.Series(group_a).value_counts(normalize=True)
    dist_b = pd.Series(group_b).value_counts(normalize=True)
    return sum(abs(dist_a.get(c, 0) - dist_b.get(c, 0)) for c in cats) / 2

# Observed TVD
missing_mask = outages['CUSTOMERS.AFFECTED.MISSING']
observed_tvd = tvd(
    outages.loc[missing_mask, 'CAUSE.CATEGORY'],
    outages.loc[~missing_mask, 'CAUSE.CATEGORY']
)

# Permutation test
n_permutations = 500
tvd_stats = []
for _ in range(n_permutations):
    shuffled = missing_mask.sample(frac=1).values
    stat = tvd(
        outages.loc[shuffled, 'CAUSE.CATEGORY'],
        outages.loc[~shuffled, 'CAUSE.CATEGORY']
    )
    tvd_stats.append(stat)

p_val_cause = np.mean(np.array(tvd_stats) >= observed_tvd)
print(f"\nMissingness vs CAUSE.CATEGORY | Observed TVD: {observed_tvd:.4f} | p-value: {p_val_cause:.4f}")

# Plot: empirical distribution of TVD permutation test
fig5 = px.histogram(
    x=tvd_stats, nbins=40,
    title='Permutation Test: Missingness of CUSTOMERS.AFFECTED vs. CAUSE.CATEGORY',
    labels={'x': 'TVD Statistic'},
    template='plotly_white',
    color_discrete_sequence=['lightblue']
)
fig5.add_vline(x=observed_tvd, line_color='red', line_dash='dash',
               annotation_text=f'Observed TVD = {observed_tvd:.3f}',
               annotation_position='top right')
fig5.update_layout(yaxis_title='Count')
fig5.show()
fig5.write_html('assets/missingness_permtest.html', include_plotlyjs='cdn')

# --- Test 2: Does missingness of CUSTOMERS.AFFECTED depend on ANOMALY.LEVEL? ---
# Test statistic: absolute difference in means (ANOMALY.LEVEL is numeric)

observed_diff = abs(
    outages.loc[missing_mask, 'ANOMALY.LEVEL'].mean() -
    outages.loc[~missing_mask, 'ANOMALY.LEVEL'].mean()
)

diff_stats = []
for _ in range(n_permutations):
    shuffled = missing_mask.sample(frac=1).values
    diff = abs(
        outages.loc[shuffled, 'ANOMALY.LEVEL'].mean() -
        outages.loc[~shuffled, 'ANOMALY.LEVEL'].mean()
    )
    diff_stats.append(diff)

p_val_anomaly = np.mean(np.array(diff_stats) >= observed_diff)
print(f"Missingness vs ANOMALY.LEVEL | Observed diff: {observed_diff:.4f} | p-value: {p_val_anomaly:.4f}")
# Expect p-value > 0.05 → missingness does NOT depend on anomaly level


Customers Affected missing rate: 42.70%

Missingness vs CAUSE.CATEGORY | Observed TVD: 0.7558 | p-value: 0.0000


Missingness vs ANOMALY.LEVEL | Observed diff: 0.0497 | p-value: 0.2080


## Step 4: Hypothesis Testing

In [24]:
subset = outages[outages['CAUSE.CATEGORY'].isin(['severe weather', 'intentional attack'])]
subset = subset[subset['CUSTOMERS.AFFECTED'].notna()]

observed_diff_hyp = (
    subset[subset['CAUSE.CATEGORY'] == 'severe weather']['CUSTOMERS.AFFECTED'].mean() -
    subset[subset['CAUSE.CATEGORY'] == 'intentional attack']['CUSTOMERS.AFFECTED'].mean()
)
print(f"\nHypothesis Test Observed Difference: {observed_diff_hyp:,.0f} customers")

# Permutation test
perm_diffs = []
for _ in range(1000):
    shuffled_labels = subset['CAUSE.CATEGORY'].sample(frac=1).values
    diff = (
        subset.loc[shuffled_labels == 'severe weather', 'CUSTOMERS.AFFECTED'].mean() -
        subset.loc[shuffled_labels == 'intentional attack', 'CUSTOMERS.AFFECTED'].mean()
    )
    perm_diffs.append(diff)

p_value_hyp = np.mean(np.array(perm_diffs) >= observed_diff_hyp)
print(f"p-value: {p_value_hyp:.4f}")

# Plot: permutation distribution
fig6 = px.histogram(
    x=perm_diffs, nbins=50,
    title='Hypothesis Test: Customers Affected — Severe Weather vs. Intentional Attack',
    labels={'x': 'Difference in Mean Customers Affected'},
    template='plotly_white',
    color_discrete_sequence=['lightgreen']
)
fig6.add_vline(x=observed_diff_hyp, line_color='red', line_dash='dash',
               annotation_text=f'Observed = {observed_diff_hyp:,.0f}',
               annotation_position='top right')
fig6.show()
fig6.write_html('assets/hypothesis_test.html', include_plotlyjs='cdn')



Hypothesis Test Observed Difference: 172,219 customers
p-value: 0.0000


## Step 5: Framing a Prediction Problem

In [25]:
print("\nClass distribution (target variable):")
print(outages['CAUSE.CATEGORY'].value_counts(normalize=True).round(3))


Class distribution (target variable):
CAUSE.CATEGORY
severe weather                   0.50
intentional attack               0.27
system operability disruption    0.08
public appeal                    0.04
equipment failure                0.04
fuel supply emergency            0.03
islanding                        0.03
Name: proportion, dtype: float64


## Step 6: Baseline Model

In [26]:
baseline_features = ['CLIMATE.REGION', 'ANOMALY.LEVEL']
target = 'CAUSE.CATEGORY'

model_df = outages[baseline_features + [target]].dropna().reset_index(drop=True)

X = model_df[baseline_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline: OneHotEncode CLIMATE.REGION, passthrough ANOMALY.LEVEL, then LogisticRegression
baseline_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['CLIMATE.REGION']),
    ('num', StandardScaler(), ['ANOMALY.LEVEL']),
])

baseline_pipeline = Pipeline([
    ('preprocessor', baseline_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, baseline_pipeline.predict(X_train))
test_acc  = accuracy_score(y_test,  baseline_pipeline.predict(X_test))
test_f1   = f1_score(y_test, baseline_pipeline.predict(X_test), average='weighted')

print(f"\nBaseline Model:")
print(f"  Train Accuracy: {train_acc:.4f}")
print(f"  Test Accuracy:  {test_acc:.4f}")
print(f"  Test Weighted F1: {test_f1:.4f}")


Baseline Model:
  Train Accuracy: 0.5683
  Test Accuracy:  0.5921
  Test Weighted F1: 0.4951


## Step 7: Final Model

In [27]:
final_features = [
    'CLIMATE.REGION',   # nominal
    'NERC.REGION',      # nominal (new)
    'SEASON',           # nominal (new — engineered from timestamp)
    'ANOMALY.LEVEL',    # quantitative
    'POPPCT_URBAN',     # quantitative (new)
    'TOTAL.PRICE',      # quantitative (new)
]

model_df2 = outages[final_features + [target]].dropna().reset_index(drop=True)
X2 = model_df2[final_features]
y2 = model_df2[target]

# Use same split proportions — re-split since we have a slightly different df
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

categorical_features = ['CLIMATE.REGION', 'NERC.REGION', 'SEASON']
numerical_features   = ['ANOMALY.LEVEL', 'POPPCT_URBAN', 'TOTAL.PRICE']

final_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('num', StandardScaler(), numerical_features),
])

final_pipeline = Pipeline([
    ('preprocessor', final_preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Hyperparameter search
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X2_train, y2_train)

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best CV F1 (weighted): {grid_search.best_score_:.4f}")

# Evaluate best model
best_model = grid_search.best_estimator_
train_f1_final = f1_score(y2_train, best_model.predict(X2_train), average='weighted')
test_f1_final  = f1_score(y2_test,  best_model.predict(X2_test),  average='weighted')
test_acc_final = accuracy_score(y2_test, best_model.predict(X2_test))

print(f"\nFinal Model:")
print(f"  Train Weighted F1: {train_f1_final:.4f}")
print(f"  Test Weighted F1:  {test_f1_final:.4f}")
print(f"  Test Accuracy:     {test_acc_final:.4f}")
print(f"\nBaseline Test F1:  {test_f1:.4f}")
print(f"Improvement:        {test_f1_final - test_f1:+.4f}")

# Optional: Confusion matrix
cm = confusion_matrix(y2_test, best_model.predict(X2_test), labels=best_model.classes_)
fig_cm = px.imshow(
    cm,
    x=best_model.classes_, y=best_model.classes_,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    title='Confusion Matrix — Final Model',
    color_continuous_scale='Blues',
    template='plotly_white',
    text_auto=True
)
fig_cm.update_layout(xaxis_tickangle=-30)
fig_cm.show()
fig_cm.write_html('assets/confusion_matrix.html', include_plotlyjs='cdn')

Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best hyperparameters: {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best CV F1 (weighted): 0.6328

Final Model:
  Train Weighted F1: 0.9211
  Test Weighted F1:  0.6420
  Test Accuracy:     0.6656

Baseline Test F1:  0.4951
Improvement:        +0.1469


## Step 8: Fairness Analysis

In [28]:
eval_df = X2_test.copy()
eval_df['true_label'] = y2_test.values
eval_df['pred_label'] = best_model.predict(X2_test)

urban_median = eval_df['POPPCT_URBAN'].median()
high_urban = eval_df['POPPCT_URBAN'] >= urban_median
low_urban  = eval_df['POPPCT_URBAN'] <  urban_median

f1_high = f1_score(eval_df.loc[high_urban, 'true_label'],
                   eval_df.loc[high_urban, 'pred_label'],
                   average='weighted')
f1_low  = f1_score(eval_df.loc[low_urban, 'true_label'],
                   eval_df.loc[low_urban, 'pred_label'],
                   average='weighted')

observed_diff_fair = abs(f1_high - f1_low)
print(f"\nFairness Analysis:")
print(f"  F1 (high urbanization): {f1_high:.4f}")
print(f"  F1 (low urbanization):  {f1_low:.4f}")
print(f"  Observed |difference|:  {observed_diff_fair:.4f}")

# Permutation test — shuffle urbanization group labels
fair_diffs = []
for _ in range(1000):
    shuffled_urban = eval_df['POPPCT_URBAN'].sample(frac=1).values >= urban_median
    f1_a = f1_score(eval_df.loc[shuffled_urban, 'true_label'],
                    eval_df.loc[shuffled_urban, 'pred_label'],
                    average='weighted')
    f1_b = f1_score(eval_df.loc[~shuffled_urban, 'true_label'],
                    eval_df.loc[~shuffled_urban, 'pred_label'],
                    average='weighted')
    fair_diffs.append(abs(f1_a - f1_b))

p_value_fair = np.mean(np.array(fair_diffs) >= observed_diff_fair)
print(f"  p-value: {p_value_fair:.4f}")

# Plot
fig7 = px.histogram(
    x=fair_diffs, nbins=40,
    title='Fairness Permutation Test: High vs. Low Urbanization States',
    labels={'x': 'Absolute Difference in Weighted F1'},
    template='plotly_white',
    color_discrete_sequence=['plum']
)
fig7.add_vline(x=observed_diff_fair, line_color='red', line_dash='dash',
               annotation_text=f'Observed = {observed_diff_fair:.3f}',
               annotation_position='top right')
fig7.show()
fig7.write_html('assets/fairness_test.html', include_plotlyjs='cdn')

print("\n✅ All steps complete!")



Fairness Analysis:
  F1 (high urbanization): 0.5704
  F1 (low urbanization):  0.7198
  Observed |difference|:  0.1494
  p-value: 0.0140



✅ All steps complete!


In [29]:
REPO_PATH = '/Users/shivamsharma0608/Desktop/power_outage_causation'
os.makedirs(f'{REPO_PATH}/assets', exist_ok=True)